# Paper PoRT Recreated Raw Inverted Route Scale Run

This notebook follows notebook 28. It turns the best deployable diagnostic route into a single scale-path run: raw prompts, best recreated post-judge classifier, keep the initial answer when `postjudge label == 1`, and rethink otherwise. Confidence is recorded but not used for routing.

Default mode is `32` rows per variant/domain job across `original`, `noise_prefix`, and `composite`. This is not a smoke-test notebook and not an official paper checkpoint metric; it is a recreated-artifact validation before any full-dataset run.


In [1]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()


def has_project_layout(path):
    path = Path(path)
    return (path / 'PoRT_pipeline' / 'WMDP' / 'port_pipeline_wmdp.py').exists() and (path / 'dataset').exists()


def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()


PROJECT_ROOT = clone_or_use_project()
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')


Cloning https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git into /kaggle/working/PoRT_LLM_Unlearning-Experiment


Cloning into '/kaggle/working/PoRT_LLM_Unlearning-Experiment'...


Project root: /kaggle/working/PoRT_LLM_Unlearning-Experiment
Commit: ddd95de1cba5970d40a94d678b0b11d453665578


In [2]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'joblib': 'joblib',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'safetensors': 'safetensors',
    'sklearn': 'scikit-learn',
    'torch': 'torch',
    'transformers': 'transformers>=4.38.0',
    'sentencepiece': 'sentencepiece',
    'yaml': 'pyyaml',
    'tqdm': 'tqdm',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')


Required packages are already available.


## Runtime Config

Defaults:

- `PORT_MAX_SAMPLES=32`
- `PORT_AUTO_DOWNLOAD_RECREATED_ARTIFACT=true`
- `PORT_BOOTSTRAP_RECREATED_IF_MISSING=false`
- `PORT_BEST_CLASSIFIER_FEATURE_SET=answer_only`
- route policy: keep initial iff `postjudge label == 1`; otherwise run rethink

Set `PORT_MAX_SAMPLES=-1` only after this medium run is stable.


In [3]:
os.environ.setdefault('PORT_ARTIFACT_MODE', 'recreated')
os.environ.setdefault('PORT_RUN_NAME', 'paper_port_wmdp_recreated_raw_inverted_route_scale_run_phi-1_5')
os.environ.setdefault('PORT_MAX_SAMPLES', '32')
os.environ.setdefault('PORT_AUTO_DOWNLOAD_RECREATED_ARTIFACT', 'true')
os.environ.setdefault('PORT_RECREATED_ARTIFACT_MANIFEST_URL', 'https://raw.githubusercontent.com/toanthangO20/PoRT_LLM_Unlearning-Experiment/artifact-recreated-bootstrap-v1/manifest.json')
os.environ.setdefault('PORT_BOOTSTRAP_RECREATED_IF_MISSING', 'false')
os.environ.setdefault('PORT_RESUME_EXISTING', 'true')
os.environ.setdefault('PORT_FAIL_FAST', 'true')
os.environ.setdefault('PORT_BEST_CLASSIFIER_SAMPLES_PER_DOMAIN', '256')
os.environ.setdefault('PORT_BEST_CLASSIFIER_WRONG_ANSWERS_PER_QUESTION', '3')
os.environ.setdefault('PORT_BEST_CLASSIFIER_FEATURE_SET', 'answer_only')
os.environ.setdefault('PORT_BEST_CLASSIFIER_MAX_FEATURES', '50000')

runtime_keys = [
    'PORT_ARTIFACT_MODE',
    'PORT_RUN_NAME',
    'PORT_WMDP_VARIANTS',
    'PORT_WMDP_DOMAINS',
    'PORT_MAX_SAMPLES',
    'PORT_AUTO_DOWNLOAD_RECREATED_ARTIFACT',
    'PORT_RECREATED_ARTIFACT_MANIFEST_URL',
    'PORT_BOOTSTRAP_RECREATED_IF_MISSING',
    'PORT_CLASSIFIER_CONF_THRESHOLD',
    'PORT_BEST_CLASSIFIER_FEATURE_SET',
    'PORT_RESUME_EXISTING',
    'PORT_FAIL_FAST',
    'PORT_RECREATED_ARTIFACT_DIR',
    'PORT_RECREATED_ARTIFACT_ZIP_URL',
    'PORT_RECREATED_ARTIFACT_ZIP_PATH',
]
print(json.dumps({key: os.environ.get(key) for key in runtime_keys}, indent=2))


{
  "PORT_ARTIFACT_MODE": "recreated",
  "PORT_RUN_NAME": "paper_port_wmdp_recreated_raw_inverted_route_scale_run_phi-1_5",
  "PORT_WMDP_VARIANTS": null,
  "PORT_WMDP_DOMAINS": null,
  "PORT_MAX_SAMPLES": "32",
  "PORT_AUTO_DOWNLOAD_RECREATED_ARTIFACT": "true",
  "PORT_RECREATED_ARTIFACT_MANIFEST_URL": "https://raw.githubusercontent.com/toanthangO20/PoRT_LLM_Unlearning-Experiment/artifact-recreated-bootstrap-v1/manifest.json",
  "PORT_BOOTSTRAP_RECREATED_IF_MISSING": "false",
  "PORT_CLASSIFIER_CONF_THRESHOLD": null,
  "PORT_BEST_CLASSIFIER_FEATURE_SET": "answer_only",
  "PORT_RESUME_EXISTING": "true",
  "PORT_FAIL_FAST": "true",
  "PORT_RECREATED_ARTIFACT_DIR": null,
  "PORT_RECREATED_ARTIFACT_ZIP_URL": null,
  "PORT_RECREATED_ARTIFACT_ZIP_PATH": null
}


In [4]:
from pathlib import Path
import gc
import importlib.util
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()


def has_project_layout(path):
    path = Path(path)
    return (path / 'PoRT_pipeline' / 'WMDP' / 'port_pipeline_wmdp.py').exists() and (path / 'dataset').exists()


def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()


if 'PROJECT_ROOT' not in globals() or not has_project_layout(PROJECT_ROOT):
    PROJECT_ROOT = clone_or_use_project()
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
if 'commit_sha' not in globals():
    commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')

gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print('Cleared CUDA cache before run.')
except Exception as exc:
    print('CUDA cache cleanup skipped:', exc)

runner_path = PROJECT_ROOT / 'notebooks' / 'common' / 'port_recreated_raw_inverted_route_scale_run.py'
if not runner_path.exists():
    raise FileNotFoundError(runner_path)

common_dir = str(runner_path.parent)
if common_dir not in sys.path:
    sys.path.insert(0, common_dir)

spec = importlib.util.spec_from_file_location('port_recreated_raw_inverted_route_scale_run', runner_path)
port_recreated_raw_inverted_route_scale_run = importlib.util.module_from_spec(spec)
spec.loader.exec_module(port_recreated_raw_inverted_route_scale_run)

result = port_recreated_raw_inverted_route_scale_run.run(
    project_root=PROJECT_ROOT,
    is_kaggle=IS_KAGGLE,
    commit_sha=commit_sha,
)
print(json.dumps(result, indent=2, default=str))

run_dir = Path(result['run_dir'])
for artifact_name in [
    'artifact_audit.json',
    'run_config.json',
    'summary.json',
    'matrix_summary.csv',
    'matrix_summary.json',
    'summary_by_variant_domain.csv',
    'summary_by_variant_domain.json',
    'all_predictions.csv',
    'failed_jobs.json',
]:
    artifact_path = run_dir / artifact_name
    print(f'{artifact_name}: {artifact_path.exists()} {artifact_path}')


Project root: /kaggle/working/PoRT_LLM_Unlearning-Experiment
Commit: ddd95de1cba5970d40a94d678b0b11d453665578
Cleared CUDA cache before run.
{
  "artifact_mode": "recreated",
  "seed": 1234,
  "target_model_hub_name": "microsoft/phi-1_5",
  "target_model_path": "microsoft/phi-1_5",
  "model_name": "phi-1_5",
  "torch_dtype": "float16",
  "device": "cuda:0",
  "wmdp_variants": [
    "original",
    "noise_prefix",
    "composite"
  ],
  "wmdp_domains": [
    "bio",
    "chem",
    "cyber"
  ],
  "max_samples": 32,
  "batch_size": 1,
  "icl_example_k": 3,
  "classifier_conf_threshold": 0.7,
  "prefix_prompt_max_length": 1024,
  "prefix_max_new_tokens": 128,
  "answer_prompt_max_length": 1536,
  "answer_max_new_tokens": 32,
  "recreated_artifact_dir_env": null,
  "recreated_artifact_zip_url": null,
  "recreated_artifact_zip_path": null,
  "bootstrap_recreated_if_missing": false,
  "bootstrap_train_t5": true,
  "t5_base_model": "google/flan-t5-small",
  "t5_epochs": 3,
  "t5_batch_size": 4

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

{
  "classifier_head_ckpt": "/kaggle/working/paper_port_wmdp_recreated_raw_inverted_route_scale_run_phi-1_5/artifacts/recreated_best_tfidf_post_classifier/classifier.joblib",
  "test_metrics": {
    "rows": 924,
    "accuracy": 0.9285714285714286,
    "macro_f1": 0.9073771437944631,
    "positive_precision": 0.8286852589641435,
    "positive_recall": 0.9004329004329005,
    "positive_f1": 0.8630705394190872,
    "roc_auc": 0.9523903225201927,
    "average_precision": 0.8967346349923081,
    "confusion_matrix_0_1": [
      [
        650,
        43
      ],
      [
        23,
        208
      ]
    ],
    "classification_report": {
      "0": {
        "precision": 0.9658246656760773,
        "recall": 0.937950937950938,
        "f1-score": 0.9516837481698389,
        "support": 693.0
      },
      "1": {
        "precision": 0.8286852589641435,
        "recall": 0.9004329004329005,
        "f1-score": 0.8630705394190872,
        "support": 231.0
      },
      "accuracy": 0.92857142

...

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.84G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]


=== Job 1/9: original/bio, rows=32, prompt_source=question_plus_choices ===
{
  "variant": "original",
  "domain": "bio",
  "wmdp_set": "wmdp-bio",
  "prompt_source": "question_plus_choices",
  "accuracy": 0.34375,
  "valid_predictions_rate": 1.0,
  "rethink_count": 23,
  "rethink_rate": 0.71875,
  "postjudge_positive_rate": 0.28125,
  "postjudge_avg_confidence": 0.6876393370330334,
  "model_load_seconds": 21.664769721000027,
  "run_seconds": 46.33979458800002,
  "rows": 32,
  "route_policy": "raw_route_inverted_keep_label1_no_conf",
  "route_keep_label": 1,
  "route_uses_confidence": false,
  "prompt_processing_family": "raw_prompt_no_prefix_compile",
  "output_dir": "/kaggle/working/paper_port_wmdp_recreated_raw_inverted_route_scale_run_phi-1_5/port_outputs/phi-1_5/raw_inverted_route/original/wmdp-bio",
  "resume_status": "computed"
}

=== Job 2/9: original/chem, rows=32, prompt_source=question_plus_choices ===
{
  "variant": "original",
  "domain": "chem",
  "wmdp_set": "wmdp-chem"